# 02 – Prepare / reset replay source
Brings the replay back to an **empty starting state**. Safe to re-run.

For each transactional table (orders, order_items, order_payments, order_reviews):
1. moves the full Olist extract from `raw` to `source` (only on the first run)
2. deletes monthly replay files (`<table>_yyyy_MM.csv`) from `raw`
3. drops the bronze table and its Auto Loader checkpoint

Reference tables (customers, products, ...) are not touched.

In [0]:
import re

dbutils.widgets.text("catalog", "olist_dev", "Target catalog")
CATALOG = dbutils.widgets.get("catalog")

LANDING = f"/Volumes/{CATALOG}/landing/raw/olist"
SOURCE = f"/Volumes/{CATALOG}/landing/source/olist"
CHECKPOINTS = f"/Volumes/{CATALOG}/bronze/checkpoints/olist"

# Transactional tables that arrive monthly; reference tables stay as one file
REPLAY_TABLES = ["orders", "order_items", "order_payments", "order_reviews"]

print(f"catalog={CATALOG}")

## Reset each transactional table
Order of steps matters: files first, then bronze and checkpoint,
so a half-finished run never leaves a checkpoint pointing to deleted files.

In [0]:
for table in REPLAY_TABLES:
    landing_dir = f"{LANDING}/{table}"
    source_dir = f"{SOURCE}/{table}"
    dbutils.fs.mkdirs(source_dir)

    # Exact pattern of files written by 03_replay_months, e.g. order_items_2017_01.csv.
    # Deleting is irreversible, so we delete ONLY files matching this pattern.
    monthly_file = re.compile(rf"^{table}_\d{{4}}_\d{{2}}\.csv$")

    deleted = 0
    for f in dbutils.fs.ls(landing_dir):
        # 1) Move full extract to source (skip if already moved -> safe to re-run)
        if f.name.startswith("olist_"):
            dbutils.fs.mv(f.path, f"{source_dir}/{f.name}")
            print(f"MOVED    {f.name} -> source/{table}/")

        # 2) Delete monthly replay files, so 03 (mode=next) starts from the beginning
        elif monthly_file.match(f.name):
            dbutils.fs.rm(f.path)
            deleted += 1

    print(f"DELETED  {deleted} monthly files in raw/{table}/")

    # 3) Drop bronze table and its checkpoint, so ingestion starts from scratch.
    #    Without dropping the checkpoint, Auto Loader would remember old file names
    #    and silently skip files with the same name when they are replayed again.
    spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.bronze.olist_{table}")
    dbutils.fs.rm(f"{CHECKPOINTS}/{table}", recurse=True)
    print(f"RESET    bronze.olist_{table} + checkpoint")